In [4]:
import os
from uuid import uuid4
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate

# -----------------------------
# CONFIG
# -----------------------------
PDF_FILE = "HR_Policy.pdf"
CHROMA_PATH = "chroma_db"
COLLECTION_NAME = "hr_collection"

# -----------------------------
# EMBEDDINGS MODEL
# -----------------------------
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

# -----------------------------
# INGESTION PIPELINE
# -----------------------------
def ingest_pdf():
    print("📥 Loading PDF...")
    loader = PyPDFLoader(PDF_FILE)
    raw_docs = loader.load()

    print(f"📄 Loaded {len(raw_docs)} pages")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_documents(raw_docs)
    print(f"🔪 Created {len(chunks)} chunks")

    vector_store = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings_model,
        persist_directory=CHROMA_PATH,
    )

    ids = [str(uuid4()) for _ in range(len(chunks))]

    print("📌 Adding chunks to Chroma...")
    vector_store.add_texts(
        texts=[doc.page_content for doc in chunks],
        metadatas=[doc.metadata for doc in chunks],
        ids=ids
    )

    vector_store.persist()
    print("✅ Ingestion complete!")


# -----------------------------
# QUERY PIPELINE
# -----------------------------
llm = ChatOpenAI(model="gpt-4o-mini")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an HR assistant. Answer only using the provided context."),
    ("user", "Context:\n{context}\n\nQuestion:\n{question}")
])

def ask_question(question):
    vector_store = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings_model,
        persist_directory=CHROMA_PATH,
    )

    q_embedding = embeddings_model.embed_query(question)
    results = vector_store.similarity_search_by_vector(q_embedding, k=5)

    if len(results) == 0:
        return "No relevant information found in the knowledge base."

    context = "\n\n".join([doc.page_content for doc in results])

    final_prompt = prompt.format(context=context, question=question)
    answer = llm.invoke(final_prompt)

    return answer.content


# -----------------------------
# RUN BOTH INGESTION + QUERY
# -----------------------------
if __name__ == "__main__":
    print("🚀 Starting ingestion...")
    ingest_pdf()

    print("\n🔍 Asking question...")
    user_question = "What is Prohibited conduct?"
    response = ask_question(user_question)

    print("\n--- ANSWER ---")
    print(response)
    print("--------------")


🚀 Starting ingestion...
📥 Loading PDF...
📄 Loaded 77 pages
🔪 Created 247 chunks
📌 Adding chunks to Chroma...
✅ Ingestion complete!

🔍 Asking question...

--- ANSWER ---
Prohibited conduct includes the following actions:

a) Repeated commitment of misdemeanors  
b) Absence from work without permission or lawful reason for three consecutive days  
c) Misuse of organization property or resources  
d) Deliberate failure and refusal to obey properly issued instructions  
e) Awarding tenders or contracts to parties that one has an interest in or is bound to gain financially or materially, or accepting gifts, kickbacks, or bribes in the form of money, services, or discounts arising from providing a service  
f) Use of GESCI time and resources for non-GESCI work  
g) Publishing or speaking and/or releasing inaccurate information about the organization  
h) Unauthorized disclosure of any confidential or classified information  
i) Pecuniary embarrassment or involvement in financial scandals tha